# 04 · Backtesting the uncertainty: prediction intervals and a solar PPA cash-flow-at-risk

Results and interpretation: [results_analysis.md](../results_analysis.md)

## 0. Settings

In [ ]:
import sys
from functools import partial
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.spines.top": False, "axes.spines.right": False})

sys.path.insert(0, str(ROOT / "src"))   # helpers shared by the notebooks
from notebook_tools import p_text, save_figure  # noqa: E402

save = partial(save_figure, folder=FIG)
import backtests  # noqa: E402
from backtests import traffic_light  # noqa: E402



# Settings. The first version of this notebook forecast the risk of day D from data up to D−1, at 95% and 99%.
# After review, each forecast is made at the end of D−2, before the day-ahead auction for D (12:00 CET on D−1),
# at the 97.5% and 99% levels at which the Basel market-risk framework backtests; short samples get exact binomial
# tests and Expected Shortfall the Acerbi–Szekely test. No setting was changed to improve a result.
INTERVAL_DIR = ROOT / "data" / "processed" / "q3_cache"   # 80% intervals stored by notebook 03, with their provenance
PERIODS = {"January–August 2026": ("2026-01-01", "2026-08-31"), "September 2026": ("2026-09-01", "2026-09-26")}
TAILS = {"below the 10% quantile": 0.10, "above the 90% quantile": 0.10, "outside the 80% interval": 0.20}
MIN_DAYS_INDEPENDENCE = 100   # Christoffersen's test needs a long sample; shorter ones get the exact binomial test only

PERFORMANCE_RATIO = 0.85      # 1 MW of solar produces 0.85 x radiation / 1000 W/m2, at most 1 MW
CFAR_LEVELS = (0.99, 0.975)   # the two backtest levels of the Basel market-risk framework
ES_LEVEL = 0.975              # its Expected Shortfall level
WINDOW = 250                  # days of history for the historical simulation
SHORT_WINDOW = 60             # a faster-reacting alternative
EWMA_LAMBDA = 0.94            # RiskMetrics decay: each earlier day weighs 6% less
AS_OF_LAG = 2                 # the forecast for day D uses data up to the end of D−2
BACKTEST = ("2024-01-01", "2026-08-31")
WINTER, SUMMER, SPRING = (11, 12, 1, 2), (5, 6, 7, 8), (3, 4, 5)
ES_SIMULATIONS, SEED = 2000, 42   # simulations for the p-value of the Acerbi–Szekely test

# the shared tests (src/backtests.py), with this notebook's settings
backtest = partial(backtests.backtest, min_days_independence=MIN_DAYS_INDEPENDENCE)
acerbi_szekely_z2 = partial(backtests.acerbi_szekely_z2, alpha=1 - ES_LEVEL, n_sim=ES_SIMULATIONS, seed=SEED)

## 1. Data

In [ ]:
con = duckdb.connect(str(ROOT / "data" / "lt_power.duckdb"), read_only=True)
lt = con.sql("""
    SELECT ts_utc, ts_local, price, solar_rad_lt_wm2
    FROM lt_hourly_all ORDER BY ts_utc
""").df()
con.close()

# prices on the auction's clock (CET delivery days x 24 hours), as in notebook 03
ts_cet = lt["ts_utc"].dt.tz_localize("UTC").dt.tz_convert("Europe/Brussels")
lt["day"], lt["hour"] = ts_cet.dt.tz_localize(None).dt.normalize(), ts_cet.dt.hour
P = lt.pivot_table(index="day", columns="hour", values="price", aggfunc="mean").reindex(columns=range(24))
REAL = P.notna()

# the most recent intervals of a full run of notebook 03
stored = sorted((p for p in INTERVAL_DIR.glob("intervals_*.pkl") if "_fast_" not in p.name), key=lambda p: p.stat().st_mtime)
if not stored:
    raise SystemExit(f"No intervals in {INTERVAL_DIR}: run notebook 03 first (with FAST = False).")
interval_store = pd.read_pickle(stored[-1])
QRA, BENCH = interval_store["result"]
print(f"Intervals: {stored[-1].name}, computed {interval_store['created']} at commit {interval_store['commit']}")
METHODS = {"QRA": QRA, "past errors": BENCH}
print(f"Prices: {P.index.min().date()} – {P.index.max().date()}; "
      f"intervals: {QRA[0.1].index.min().date()} – {QRA[0.1].index.max().date()}")

## 2. Backtesting the 80% prediction intervals

In [ ]:
def hit_table(quantiles, start, end, tail):
    # Daily hits for every delivery hour (days x 24), NaN where there is no price or no interval.
    days = pd.date_range(start, end, freq="D")
    price, low, high = P.reindex(days), quantiles[0.1].reindex(days), quantiles[0.9].reindex(days)
    valid = price.notna() & low.notna() & high.notna()
    hit = {"below the 10% quantile": price < low, "above the 90% quantile": price > high,
           "outside the 80% interval": (price < low) | (price > high)}[tail]
    return hit.astype(float).where(valid)


per_hour, pooled = [], []
for method, quantiles in METHODS.items():
    for period, (start, end) in PERIODS.items():
        for tail, p in TAILS.items():
            hits = hit_table(quantiles, start, end, tail)
            for h in range(24):
                series = hits[h].dropna()
                if len(series) >= 20:
                    per_hour.append({"method": method, "period": period, "tail": tail, "hour": h, **backtest(series, p)})
            # all hours together: hit rate against p, errors clustered by day (the 24 hours share one auction)
            stacked = hits.rename_axis(index="day", columns="hour").stack().dropna().rename("hit").reset_index()
            fit = sm.OLS(stacked["hit"] - p, np.ones(len(stacked))).fit(
                cov_type="cluster", cov_kwds={"groups": stacked["day"].factorize()[0]})
            pooled.append({"method": method, "period": period, "tail": tail, "hours": len(stacked),
                           "hit rate": stacked["hit"].mean(), "expected": p, "p-value (clustered by day)": fit.pvalues.iloc[0]})
per_hour, pooled = pd.DataFrame(per_hour), pd.DataFrame(pooled)
display(pooled.round(3))

In [ ]:
# per-hour results: how many of the 24 delivery hours fail each test at the 5% level
# (with 24 tests, about 1.2 rejections are expected by chance alone; September's 26 days are too few for Christoffersen)
groups = ["method", "period", "tail"]
summary_hours = (per_hour.assign(**{
        "exact binomial rejects": per_hour["binomial p"] < 0.05,
        "Kupiec rejects": per_hour["Kupiec p"] < 0.05,
        "Christoffersen rejects": per_hour["Christoffersen p"] < 0.05,
        "conditional coverage rejects": per_hour["conditional coverage p"] < 0.05,
        "yellow zone": per_hour["zone"] == "yellow", "red zone": per_hour["zone"] == "red"})
    .groupby(groups)
    [["exact binomial rejects", "Kupiec rejects", "Christoffersen rejects", "conditional coverage rejects",
      "yellow zone", "red zone"]].sum()
    .astype(int))
summary_hours["hours tested"] = per_hour.groupby(groups).size()
summary_hours["hours with an independence test"] = per_hour.groupby(groups)["Christoffersen p"].count()
summary_hours

In [ ]:
qra = per_hour[(per_hour["method"] == "QRA") & (per_hour["period"] == "January–August 2026")]
fig, ax = plt.subplots(figsize=(10, 4.2))
for tail, color in [("below the 10% quantile", "#4c72b0"), ("above the 90% quantile", "#dd8452")]:
    rows = qra[qra["tail"] == tail].sort_values("hour")
    ax.plot(rows["hour"], 100 * rows["hit rate"], marker="o", ms=4, color=color, label=f"price {tail}")
ax.axhline(10, color="dimgray", ls="--", lw=1)
ax.text(23.5, 9.6, "target 10%", color="dimgray", ha="right", va="top", fontsize=10, fontweight="bold")
ax.set_ylim(bottom=0)
ax.set(xlabel="Delivery hour (CET)", ylabel="Share of days (%)",
       title="How often the price fell outside each side of the 80% interval, January–August 2026")
ax.legend()
fig.tight_layout()
save(fig, "q4_interval_hits_by_hour")
plt.show()

# the share of hours outside the interval, day by day and as a 14-day rolling mean
days = pd.date_range(PERIODS["January–August 2026"][0], PERIODS["September 2026"][1], freq="D")
outside = hit_table(QRA, days[0], days[-1], "outside the 80% interval").mean(axis=1)
fig, ax = plt.subplots(figsize=(10, 4.6))
bars = ax.bar(outside.index, 100 * outside, color="#c9d6ea", width=1.0, label="Daily share of hours outside the interval (QRA)")
mean_line, = ax.plot(outside.index, 100 * outside.rolling(14, min_periods=7).mean(), color="#4c72b0", lw=2, label="14-day mean")
target = ax.axhline(20, color="grey", ls="--", lw=1, label="Target: 20% of hours")
clean_test = ax.axvline(pd.Timestamp(PERIODS["September 2026"][0]), color="black", lw=0.8, label="September 1: start of the clean test")
ax.set(ylabel="Hours outside (%)", title="Misses of the 80% prediction interval, 2026")
ax.legend(handles=[bars, mean_line, target, clean_test], loc="upper center", bbox_to_anchor=(0.5, -0.1),
          ncol=2, frameon=False, fontsize=9)
fig.tight_layout()
save(fig, "q4_interval_misses_over_time")
plt.show()

## 3. A solar PPA position: daily cash-flow-at-risk and Expected Shortfall

In [ ]:
# Daily cash-flow-at-risk (CFaR): a quantile of the realized daily settlement of the position. It is not a
# market-risk VaR, which would revalue the PPA off a forward curve and measure the change in its value.
lt["date"] = lt["ts_local"].dt.normalize()


def ppa_cashflow(performance_ratio=PERFORMANCE_RATIO, fixed_price=None):
    # Daily cash flow of buying the output of 1 MW of solar at a fixed price: output x (day-ahead price − fixed price).
    # The fixed price defaults to the 2023 capture price of the same output profile.
    hours = lt.assign(solar_mwh=(performance_ratio * lt["solar_rad_lt_wm2"] / 1000).clip(0, 1))
    hours = hours.dropna(subset=["price", "solar_mwh"])
    if fixed_price is None:
        y2023 = hours[hours["date"].dt.year == 2023]
        fixed_price = (y2023["solar_mwh"] * y2023["price"]).sum() / y2023["solar_mwh"].sum()
    flow = (hours["solar_mwh"] * (hours["price"] - fixed_price)).groupby(hours["date"]).sum()
    return flow, fixed_price


cash, K = ppa_cashflow()
print(f"PPA price K = {K:.1f} €/MWh (2023 capture price of a 1 MW solar profile); "
      f"daily cash flows from {cash.index.min().date()} to {cash.index.max().date()}")
display(cash.groupby(cash.index.year).describe()[["mean", "std", "min", "max"]].round(1))

In [ ]:
MODELS = {"HS": f"HS {WINDOW} days", "HS short": f"HS {SHORT_WINDOW} days", "FHS": "Filtered HS (EWMA)"}


def level_name(level):
    return f"{100 * level:g}%"


def ewma_sigma(x, lam=EWMA_LAMBDA, warmup=30):
    # RiskMetrics volatility of x for each day, using only the days before it.
    sigma2 = np.empty(len(x))
    sigma2[0] = np.var(x[:warmup])
    for t in range(1, len(x)):
        sigma2[t] = lam * sigma2[t - 1] + (1 - lam) * x[t - 1] ** 2
    return np.sqrt(np.maximum(sigma2, 1e-12))


def tail_loss(sample, level):
    # CFaR and ES at `level` from a sample of daily cash flows, as positive losses.
    q = np.quantile(sample, 1 - level)
    return -q, -sample[sample <= q].mean()


def cfar_forecasts(cash, window=WINDOW, short_window=SHORT_WINDOW, lam=EWMA_LAMBDA, lag=AS_OF_LAG, keep_samples=False):
    # For every backtest day D: CFaR and ES of each model from the cash flows up to the end of D − lag.
    values, dates = cash.to_numpy(), cash.index
    sigma = ewma_sigma(values, lam)            # sigma[t]: volatility forecast for day t from the days before t
    standardised = values / sigma
    first = np.searchsorted(dates, pd.Timestamp(BACKTEST[0]))
    last = np.searchsorted(dates, pd.Timestamp(BACKTEST[1]), side="right")
    rows, samples = [], {model: [] for model in MODELS}
    for t in range(first, last):
        h = t - lag + 1                         # the history ends at day t − lag
        row = {"date": dates[t], "cash flow": values[t]}
        # a RiskMetrics forecast is flat over the horizon: the forecast for D made at D − 2 equals the one for D − 1
        for model, sample, scale in [("HS", values[h - window:h], 1.0),
                                     ("HS short", values[h - short_window:h], 1.0),
                                     ("FHS", standardised[h - window:h], sigma[h])]:
            for level in CFAR_LEVELS:
                cfar, es = tail_loss(sample, level)
                row[f"{model} CFaR {level_name(level)}"] = scale * cfar
                if level == ES_LEVEL:
                    row[f"{model} ES {level_name(level)}"] = scale * es
            if keep_samples:
                samples[model].append(scale * sample)
        rows.append(row)
    risk = pd.DataFrame(rows).set_index("date")
    return (risk, samples) if keep_samples else risk


risk, samples = cfar_forecasts(cash, keep_samples=True)
print(f"{len(risk)} backtest days, {risk.index.min().date()} – {risk.index.max().date()}; "
      f"each forecast uses data up to the end of the day {AS_OF_LAG} days before delivery")

## 4. Backtesting the cash-flow-at-risk models

In [ ]:
results, es_tests = [], []
for model, label in MODELS.items():
    for level in CFAR_LEVELS:
        hits = (risk["cash flow"] < -risk[f"{model} CFaR {level_name(level)}"]).astype(int)
        res = backtest(hits.to_numpy(), 1 - level)
        res["zones by year"] = ", ".join(f"{year}: {traffic_light(h.sum(), len(h), 1 - level)}"
                                         for year, h in hits.groupby(hits.index.year))
        results.append({"model": label, "level": level_name(level), **res})
    z2, p_value = acerbi_szekely_z2(risk["cash flow"], risk[f"{model} CFaR {level_name(ES_LEVEL)}"],
                                    risk[f"{model} ES {level_name(ES_LEVEL)}"], samples[model])
    es_tests.append({"model": label, "Z2": z2, "p-value": p_value})
cfar_results, es_tests = pd.DataFrame(results), pd.DataFrame(es_tests)
display(cfar_results.drop(columns="expected").round(3))
display(es_tests.round(3))

seasonal = []
for model, label in MODELS.items():
    for season, months in [("winter (Nov–Feb)", WINTER), ("summer (May–Aug)", SUMMER)]:
        part = risk[risk.index.month.isin(months)]
        limit = part[f"{model} CFaR 99%"]
        seasonal.append({"model": label, "season": season, "days": len(part),
                         "exceptions at 99%": int((part["cash flow"] < -limit).sum()),
                         "average CFaR 99%, €": limit.mean(),
                         "average absolute cash flow, €": part["cash flow"].abs().mean()})
pd.DataFrame(seasonal).round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.8))
bars = ax.bar(risk.index, risk["cash flow"], color="#bbbbbb", width=1.0, label=f"Daily cash flow, 1 MW of solar bought at {K:.1f} €/MWh")
handles = [bars, plt.Line2D([], [], linestyle="none", label=" ")]   # an empty slot keeps each model in its own column
for model, name, color in [("HS", f"historical simulation ({WINDOW} days)", "#dd8452"),
                           ("FHS", "EWMA-filtered simulation", "#4c72b0")]:
    limit = -risk[f"{model} CFaR 99%"]
    line, = ax.plot(risk.index, limit, color=color, lw=1.4, label=f"−CFaR 99%, {name}")
    breaks = risk["cash flow"] < limit
    dots = ax.scatter(risk.index[breaks], risk.loc[breaks, "cash flow"], color=color, s=18, zorder=3,
                      label=f"Loss beyond CFaR, one dot per day ({int(breaks.sum())} days)")
    handles += [line, dots]
ax.axhline(0, color="black", lw=0.6)
ax.set(ylabel="€ per day", title="Solar PPA, daily cash flow and 99% cash-flow-at-risk")
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3, frameon=False, fontsize=9)
fig.tight_layout()
save(fig, "q4_ppa_cfar_backtest")
plt.show()

## 5. Sensitivity analysis

In [ ]:
# One setting changed at a time; 99% CFaR exceptions over the same 974 backtest days (about 10 expected)
variants = [("base case", {}, {}),
            ("PPA price 70 €/MWh", {"fixed_price": 70.0}, {}),
            ("PPA price 100 €/MWh", {"fixed_price": 100.0}, {}),
            ("performance ratio 0.75", {"performance_ratio": 0.75}, {}),
            ("performance ratio 0.95", {"performance_ratio": 0.95}, {}),
            ("history 125 days", {}, {"window": 125}),
            ("history 360 days", {}, {"window": 360}),
            ("EWMA λ = 0.90", {}, {"lam": 0.90}),
            ("EWMA λ = 0.97", {}, {"lam": 0.97})]
rows = []
for name, flow_args, model_args in variants:
    flows, _ = ppa_cashflow(**flow_args)
    r = cfar_forecasts(flows, **model_args)
    row = {"variant": name}
    for model, label in [("HS", "historical simulation"), ("FHS", "filtered HS (EWMA)")]:
        x = int((r["cash flow"] < -r[f"{model} CFaR 99%"]).sum())
        row[f"{label}: exceptions"] = x
        row[f"{label}: zone"] = traffic_light(x, len(r), 0.01)
    rows.append(row)
sensitivity = pd.DataFrame(rows).set_index("variant")
sensitivity

## 6. Stress scenario: a sunny spring week

In [ ]:
# Historical stress: the complete spring week (Monday–Sunday, March–May) with the largest cash-flow loss in the backtest
weeks = risk["cash flow"].groupby(risk.index.to_period("W-SUN")).agg(["sum", "count"])
spring_weeks = weeks[(weeks["count"] == 7) & weeks.index.start_time.month.isin(SPRING)]
worst = spring_weeks["sum"].idxmin()
week = risk[(risk.index >= worst.start_time) & (risk.index <= worst.end_time)]

# Hypothetical stress: that week's hourly prices with the output of the sunniest spring week on record, hour by hour
hourly = lt.dropna(subset=["price", "solar_rad_lt_wm2"]).assign(
    solar_mwh=lambda d: (PERFORMANCE_RATIO * d["solar_rad_lt_wm2"] / 1000).clip(0, 1),
    week=lambda d: d["ts_local"].dt.to_period("W-SUN"),
    slot=lambda d: d["ts_local"].dt.dayofweek * 24 + d["ts_local"].dt.hour)
spring_output = (hourly[hourly["ts_local"].dt.month.isin(SPRING)]
                 .groupby("week")["solar_mwh"].agg(["sum", "count"]))
sunniest = spring_output[spring_output["count"] >= 160]["sum"].idxmax()   # complete weeks only
prices = hourly[hourly["week"] == worst].groupby("slot")["price"].mean()
output = hourly[hourly["week"] == sunniest].groupby("slot")["solar_mwh"].mean()
hypothetical = float((output * (prices - K)).dropna().sum())

stress = pd.Series({
    "worst spring week": f"{worst.start_time.date()} – {worst.end_time.date()}",
    "its cash flow, €": round(week["cash flow"].sum()),
    "its worst day, €": round(week["cash flow"].min()),
    "days beyond the 99% CFaR, historical simulation": int((week["cash flow"] < -week["HS CFaR 99%"]).sum()),
    "days beyond the 99% CFaR, filtered HS": int((week["cash flow"] < -week["FHS CFaR 99%"]).sum()),
    "sum of the seven daily 99% CFaR, historical simulation, €": round(week["HS CFaR 99%"].sum()),
    "sum of the seven daily 99% CFaR, filtered HS, €": round(week["FHS CFaR 99%"].sum()),
    "sunniest spring week (output used for the hypothetical case)": f"{sunniest.start_time.date()} – {sunniest.end_time.date()}",
    "hypothetical: the worst week's prices with the sunniest week's output, €": round(hypothetical),
}, name="value")
stress.to_frame()

## 7. Summary

In [ ]:
rows = []
for period in PERIODS:
    for tail in TAILS:
        r = pooled[(pooled["method"] == "QRA") & (pooled["period"] == period) & (pooled["tail"] == tail)].iloc[0]
        h = summary_hours.loc[("QRA", period, tail)]
        rows.append([f"Intervals (QRA), {period}: price {tail}",
                     f"{100 * r['hit rate']:.1f}% of hours against {100 * r['expected']:.0f}% "
                     f"(p = {p_text(r['p-value (clustered by day)'])}); the exact binomial test rejects in "
                     f"{h['exact binomial rejects']} of {h['hours tested']} hours, Christoffersen in "
                     f"{h['Christoffersen rejects']} of {h['hours with an independence test']}"])
for _, r in cfar_results.iterrows():
    rows.append([f"Solar PPA, {r['model']}, CFaR {r['level']}",
                 f"{r['hits']} exceptions in {r['days']} days ({100 * r['hit rate']:.1f}% against "
                 f"{100 * r['expected']:.1f}%); binomial p = {p_text(r['binomial p'])}, Christoffersen p = "
                 f"{p_text(r['Christoffersen p'])}; zones {r['zones by year']}"])
for _, r in es_tests.iterrows():
    rows.append([f"Solar PPA, {r['model']}, ES 97.5% (Acerbi–Szekely Z2)", f"Z2 = {r['Z2']:.2f}, p = {p_text(r['p-value'])}"])
summary = pd.DataFrame(rows, columns=["finding", "estimate"])
summary.to_csv(FIG / "q4_summary.csv", index=False)
pd.set_option("display.max_colwidth", 200)
summary